# garimpo-uncertainty on Google Colab

This notebook reproduces the experiment of the dissertation on Google Colab. It tests the code on a
GPU, downloads the Sentinel-2 data, runs a pilot on 1% of the training tiles, runs the four models
on all of them, and merges their result tables.

Each cell runs one command, so you can run a cell that failed again on its own.

## How to use it

1. Once, follow "Before the first session".
2. In every session, run the cells of "Session setup" in order.
3. Then run the cells of one step, in order, on the runtime the table below gives.

Colab queues the cells you start while another one runs, so you can start a whole step at once.
Keep the tab open until it ends. Do not use Runtime > Run all: the last cell of each step writes
the pending files to Drive and releases the runtime, and the cells after it would have no machine to run on.

| Step | Runtime | When | Saved in Drive |
|---|---|---|---|
| 1. Tests | GPU | Once, before the first paid run | Nothing |
| 2. Data | CPU | Once | `data.tar.zst` |
| 3. Pilot | L4 GPU | Until all three variants pass | `runs/pilot/`, `logs/` |
| 4. Full run | GPU | Once per model, one model per session | `runs/<model>-f1.0/`, `logs/` |
| 5. Merge | CPU | Once, after the four full runs pass | `outputs/` |

Choose the runtime in Runtime > Change runtime type before the session setup. Changing it later
starts a new session, and the setup has to run again.

## Before the first session

The repository is private, and each session clones it with a GitHub token that can only read it.

1. On GitHub, create a fine-grained personal access token for this repository only, with the
   permission Contents: Read-only.
2. In Colab, open Secrets (the key icon on the left), add a secret named `GH_TOKEN` with the token
   as its value, and turn on Notebook access.
3. Open this notebook with File > Upload notebook.

Since the token cannot push, change the code on your own machine and push it from there. The next
session clones the new version.

## Session setup

The runtime's disk is erased when a session ends. These cells clone the code again, build the
Python environment and copy the data and the earlier runs back from Drive.

Mount Google Drive. Colab asks for permission the first time. Everything that must outlive the
session is kept in `MyDrive/garimpo`.

In [ ]:
# @title Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

The paths and variables the other cells use. `DATA` must stay `/content/garimpo-data`: every run
records the path of its data and refuses to continue when it is restored under another path.

In [ ]:
# @title Settings
import json
import os
import shutil
import subprocess
import threading
import time
from pathlib import Path

from google.colab import runtime, userdata

REPO = "github.com/meiazero/garimpo-uncertainty.git"
CODE = "/content/garimpo-uncertainty"  # the clone; erased with the session
DRIVE = "/content/drive/MyDrive/garimpo"  # kept between sessions
DATA = "/content/garimpo-data"  # runs record this path, so it never changes
TOKEN = userdata.get("GH_TOKEN")

os.environ["GARIMPO_DATA_DIR"] = DATA
periodic_copy = None  # the copy to Drive every 30 minutes of step 4, once started

The functions the steps use. Every pilot and full-run job ends by printing `PIPELINE CHECK: PASS`
or `PIPELINE CHECK: FAIL`, and a cell keeps going when one of its shell commands fails, so read
that line to know how a job went. A job's log keeps every attempt: `start_log` marks where a new
one begins, and `print_verdicts` prints the verdict of the last attempt only.
`copy_to_drive_every_30_min` saves the runs and logs during a long job. `release_runtime` ends each
step: it writes the files still pending to Drive, then releases the machine. The cells before it in
a step do not raise, so a queued step always reaches it.

In [ ]:
# @title Define the helper functions
VERDICTS = ("PIPELINE CHECK: PASS", "PIPELINE CHECK: FAIL")
ATTEMPT = "=== attempt started "


def start_log(name: str) -> None:
    """Mark the start of a new attempt in the job's log. It never raises."""
    started = time.strftime("%Y-%m-%d %H:%M", time.gmtime())
    try:
        with (Path("outputs/logs") / f"{name}.log").open("a", encoding="utf-8") as log:
            log.write(f"{ATTEMPT}{started} UTC ===\n")
    except OSError as error:
        print(f"Could not mark the attempt in the log: {error}")


def print_verdicts(log_names: list[str]) -> None:
    """Print the pipeline-check line of the last attempt in each log. It never raises."""
    for name in log_names:
        log = Path("outputs/logs") / f"{name}.log"
        text = log.read_text(encoding="utf-8", errors="replace") if log.exists() else ""
        text = text.rsplit(ATTEMPT, 1)[-1]
        found = [line for line in text.splitlines() if line.startswith(VERDICTS)]
        print(name, found[-1] if found else "NO VERDICT: the job or its check died; see the log")


def copy_to_drive_every_30_min(stop: threading.Event) -> None:
    """Copy runs/ and the logs to Drive every 30 minutes until stop is set."""
    runs = ["rsync", "-a", "--exclude", "*.tmp", "runs/", f"{DRIVE}/runs/"]
    logs = ["rsync", "-a", "outputs/logs/", f"{DRIVE}/logs/"]
    while not stop.wait(1800):
        for command in (runs, logs):
            try:
                copied = subprocess.run(command, check=False)  # noqa: S603
            except OSError as error:
                print(f"The copy to Drive every 30 minutes failed: {error}")
                continue
            if copied.returncode != 0:
                print(f"The copy to Drive every 30 minutes ended with code {copied.returncode}")


def release_runtime() -> None:
    """Write pending files to Drive, then release the runtime, which ends the session."""
    try:
        drive.flush_and_unmount()
    except Exception as error:  # noqa: BLE001 -- the runtime must be released whatever failed
        print(f"Drive did not confirm the last writes; check the files in Drive: {error}")
    runtime.unassign()

`rsync` copies the runs to and from Drive, and `zstd` packs and unpacks the data archive. The
pipeline compares each input file's size and modification time to tell which stages are already
done, and both `rsync -a` and `tar` keep those times.

In [ ]:
# @title Install rsync and zstd
!apt-get -qq install -y rsync zstd > /dev/null

`uv` builds the Python environment from `uv.lock`, which pins every package version.

In [ ]:
# @title Install uv
!pip install -q uv

Clone the code with its `compass/` submodule, the adapted COMPASS method. Run again in the same
session, this cell says the folder already exists, and nothing changes.

In [ ]:
# @title Clone the code
!git clone --recurse-submodules https://{TOKEN}@{REPO} {CODE}

In [ ]:
# @title Enter the code folder
%cd {CODE}

In [ ]:
# @title Build the Python environment
!uv sync

The commit this session runs. Note it next to any result you keep.

In [ ]:
# @title Show the commit
!git log -1 --format='%h %cs %s'

In [ ]:
# @title Create the folders
!mkdir -p {DRIVE}/runs {DRIVE}/logs {DATA} runs outputs/logs

Unpack the data archive that step 2 writes. The unpacked store always takes 69.2 GB, whatever the
training fraction, because its arrays are created for all 18,864 cells. Skip this cell in steps 1
and 5, which do not read the data. Before step 2 has run, `tar` reports that the archive is
missing.

In [ ]:
# @title Unpack the data
!tar -I zstd -xf {DRIVE}/data.tar.zst -C {DATA}

Copy back the runs and logs of earlier sessions. The finished stages of a run are then skipped,
and each job's log keeps the earlier attempts above the new one, including one cut short by a
session that ended. If one of these two cells fails,
fix it before running a step: a step that runs without its earlier runs starts them over, and its
copy to Drive replaces the saved ones.

In [ ]:
# @title Restore the runs
!rsync -a {DRIVE}/runs/ runs/

In [ ]:
# @title Restore the logs
!rsync -a {DRIVE}/logs/ outputs/logs/

Free disk. The store needs 69.2 GB and the Python environment 6.9 GB, plus room for the runs.

In [ ]:
# @title Show the free disk
!df -h /content

## Step 1. Tests

Runtime: GPU. Once, before the first paid run.

The tests build a small synthetic dataset of their own, so they need no download and you can skip
the unpack cell. GitHub's CI runs them on a CPU; here they run on CUDA. A few tests skip on a
GPU on purpose, such as those that compare a result with one computed on the CPU, and every other
test should pass. Their files stay in `tests/artifacts/` until the session ends, so read any
failure before you release the GPU.

In [ ]:
# @title Run the tests on the GPU
!GARIMPO_TEST_DEVICE=cuda uv run --no-sync pytest -q

In [ ]:
# @title Write pending files to Drive and release the GPU
release_runtime()

## Step 2. Data

Runtime: CPU. Once.

These cells download what the runs read and pack it into one archive in Drive, which the session
setup unpacks. If an archive already exists, run the unpack cell first: the downloads then resume
from it, and the new archive keeps everything the old one had.

A session that ends during a download loses whatever is not yet archived. To see how long a cell
took, hover over the icon to its left. The time of the 1% download tells you whether the 100% one
fits in a session.

The labels and tile metadata of the mining dataset (Jasansky et al., Zenodo, 2024).

In [ ]:
# @title Download the metadata
!uv run --no-sync python scripts/download_data.py metadata

The Sentinel-2 scenes of the tiles the pilot uses, cut into cells of 512 × 512 pixels: 10.9 GB.
`--resampling-20m bilinear` sets how bands B11 and B12 reach the 10 m grid.

A tile that fails does not stop the download. The command ends with `done: N tiles in ..., K not
ok` and keeps each failure's reason. Run the cell again to retry only the tiles that are not ok.
An industrial tile whose scene does not exist keeps the status `missing_item`, and the runs
proceed without it. A missing artisanal tile fails the data check at the start of every run.

In [ ]:
# @title Download the 1% scenes
!uv run --no-sync python scripts/download_data.py scenes \
    --tiles outputs/tiles/tiles_1pct.json --resampling-20m bilinear

All 1,179 tiles, 69.2 GB. The 1% tiles are among them and are not downloaded again.

In [ ]:
# @title Download the 100% scenes
!uv run --no-sync python scripts/download_data.py scenes \
    --tiles outputs/tiles/tiles_100pct.json --resampling-20m bilinear

The ImageNet and Sentinel-2 MoCo starting weights of the U-Net encoder.

In [ ]:
# @title Download the starting weights
!uv run --no-sync python scripts/download_data.py weights \
    --encoder-weights imagenet sentinel2_moco

Pack the store into `data.tar.zst` in Drive. The cell first reads the status of the last scenes
download and archives nothing unless it is `ok`. `tar` then writes a temporary file, which
replaces the old archive only when `tar` succeeds. Either way, a failed download or archive leaves
the old archive as it was. If the unpack cell of the session setup failed, do not run this cell:
the status it reads comes from the old archive and says `ok` even for a store cut short. zstd runs at level 3 because higher levels compress much more slowly on
the runtime's two CPUs.

In [ ]:
# @title Archive the data in Drive
def archive_data() -> None:
    """Pack the store into Drive, keeping the old archive unless the download and tar succeed."""
    archive = Path(DRIVE) / "data.tar.zst"
    partial = Path(DRIVE) / "data.tar.zst.tmp"
    status_file = Path(DATA) / "status" / "download_scenes.json"
    status = "not run"
    if status_file.exists():
        status = json.loads(status_file.read_text(encoding="utf-8"))["status"]
    if status != "ok":
        print(f"Not archived: the last scenes download is {status!r}. {archive} is unchanged.")
        return
    tar = [shutil.which("tar"), "--format=posix", "-I", "zstd -3 -T0", "-cf", partial, "-C", DATA]
    packed = subprocess.run([*tar, "."], capture_output=True, text=True, check=False)  # noqa: S603
    if packed.returncode != 0:
        partial.unlink(missing_ok=True)
        print(f"tar failed, {archive} is unchanged:\n{packed.stderr}")
        return
    partial.replace(archive)
    print(f"Archived in {archive}")


try:
    archive_data()
except (OSError, ValueError, KeyError) as error:  # must not stop the queue before the release
    print(f"Archiving failed: {error}")

In [ ]:
# @title Write pending files to Drive and release the runtime
release_runtime()

## Step 3. Pilot

Runtime: L4 GPU. Until all three variants pass.

The pilot runs the whole pipeline on 1% of the training tiles, with validation, calibration and
test whole, to find bugs on a GPU before the paid runs. Its numbers say nothing about the methods.
Each U-Net starting point has its own cell: `none` (from scratch), `imagenet` and
`sentinel2_moco`. Each one writes `runs/pilot/unet-<weights>/` and the log
`outputs/logs/pilot-<weights>.log`.

Each pilot cell has two options, used by that cell only. Leave both empty on the first run. After a
code fix, push it, start a new session, and set `FORCE_FROM` to the first stage the fix touches: that stage and the ones after it run again, and
the earlier ones are skipped. The stages, in order:

`train`, `infer`, `fit_directions`, `layer_monotonicity`, `choose_layer`, `endpoint_monotonicity`,
`compass_scores`, `calibrate`, `generate_intervals`, `compute_metrics`

`FORCE_FROM=train` refuses while `last.pt` holds a training that has not finished; leave the
option empty to resume that training instead. Set `FORCE_VALIDATE` to `1` to check the data again.

In [ ]:
# @title Pilot from scratch
FORCE_FROM = ""  # @param ["", "train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = ""  # @param ["", "1"]
start_log("pilot-none")
!FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={FORCE_VALIDATE} ENCODER_WEIGHTS=none \
    bash slurm/pilot.sbatch 2>&1 | tee -a outputs/logs/pilot-none.log

In [ ]:
# @title Pilot from ImageNet weights
FORCE_FROM = ""  # @param ["", "train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = ""  # @param ["", "1"]
start_log("pilot-imagenet")
!FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={FORCE_VALIDATE} ENCODER_WEIGHTS=imagenet \
    bash slurm/pilot.sbatch 2>&1 | tee -a outputs/logs/pilot-imagenet.log

In [ ]:
# @title Pilot from Sentinel-2 MoCo weights
FORCE_FROM = ""  # @param ["", "train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = ""  # @param ["", "1"]
start_log("pilot-sentinel2_moco")
!FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={FORCE_VALIDATE} ENCODER_WEIGHTS=sentinel2_moco \
    bash slurm/pilot.sbatch 2>&1 | tee -a outputs/logs/pilot-sentinel2_moco.log

In [ ]:
# @title Save the runs in Drive
!rsync -a runs/ {DRIVE}/runs/

In [ ]:
# @title Save the logs in Drive
!rsync -a outputs/logs/ {DRIVE}/logs/

One line per variant. `PASS` means every stage finished and every check held. After `FAIL`, the
lines above it in the log name the stage that failed and its error. `NO VERDICT` means the job or
its check died first, and the log shows where.

In [ ]:
# @title Print the verdicts
print_verdicts(["pilot-none", "pilot-imagenet", "pilot-sentinel2_moco"])

In [ ]:
# @title Write pending files to Drive and release the GPU
release_runtime()

## Step 4. Full run

Runtime: GPU. One model per session, four in all.

Start once the pilot passes. The four models all use a ResNet-18 encoder and train on 100% of the
training tiles:

| `ENV` | Model |
|---|---|
| `unet_scratch-f1.0` | U-Net from scratch |
| `unet_imagenet-f1.0` | U-Net from ImageNet weights |
| `unet_sentinel2-f1.0` | U-Net from Sentinel-2 MoCo weights |
| `convnet-f1.0` | DeepLabV3 from segmentation-models-pytorch |

Each model reads its values from `slurm/<ENV>.env`. On your own machine, copy
`slurm/full.env.example` to that name, fill in every value, with `RUN_DIR=runs/<ENV>`, then commit
and push it. `docs/decisions.md` gives the values already fixed for
the dissertation, with their sources, and lists those still open with what settles each.

The run chooses the COMPASS-J layer on the validation set and writes the reason to
`chosen_layer.json`. If the session ends during the run, start a new one, run the setup, and run
this step again with the same `ENV` and `FORCE_FROM` empty: finished stages are skipped and
training continues from its last checkpoint. The options of the run cell work as in the pilot.

Copy `runs/` and the logs to Drive every 30 minutes while the model runs, so a session that ends
early keeps the finished stages, the last checkpoint and the log up to that point. Running this
cell again while the copy runs starts nothing new.

In [ ]:
# @title Start the copy to Drive every 30 minutes
if periodic_copy is None or not periodic_copy.is_alive():
    stop_copy = threading.Event()
    periodic_copy = threading.Thread(
        target=copy_to_drive_every_30_min, args=(stop_copy,), daemon=True
    )
    periodic_copy.start()

In [ ]:
# @title Run the model
ENV = "unet_scratch-f1.0"  # @param ["unet_scratch-f1.0", "unet_imagenet-f1.0", "unet_sentinel2-f1.0", "convnet-f1.0"]
FORCE_FROM = ""  # @param ["", "train", "infer", "fit_directions", "layer_monotonicity", "choose_layer", "endpoint_monotonicity", "compass_scores", "calibrate", "generate_intervals", "compute_metrics"]
FORCE_VALIDATE = ""  # @param ["", "1"]
start_log(ENV)
!NUM_WORKERS={os.cpu_count()} FORCE_FROM={FORCE_FROM} FORCE_VALIDATE={FORCE_VALIDATE} \
    bash slurm/full.sbatch slurm/{ENV}.env 2>&1 | tee -a outputs/logs/{ENV}.log

Stop the periodic copy. The cell waits for a copy in progress to finish, so that it and the final
copy never write to Drive at the same time.

In [ ]:
# @title Stop the copy every 30 minutes
if periodic_copy is not None:
    stop_copy.set()
    periodic_copy.join()

In [ ]:
# @title Save the runs in Drive
!rsync -a runs/ {DRIVE}/runs/

In [ ]:
# @title Save the logs in Drive
!rsync -a outputs/logs/ {DRIVE}/logs/

In [ ]:
# @title Print the verdict
print_verdicts([ENV])

In [ ]:
# @title Write pending files to Drive and release the GPU
release_runtime()

## Step 5. Merge

Runtime: CPU. Once, after the four full runs pass.

The session setup has copied the four runs back from Drive; the unpack cell is not needed. The
merge writes `results.csv` and the comparison tables to `outputs/`. Check that it ended without an
error before you copy the tables to Drive. Then download `MyDrive/garimpo/outputs/` and commit it
from your own machine: these are the tables the dissertation reports.

In [ ]:
# @title Merge the four runs
!uv run --no-sync python scripts/merge_results.py --out outputs --runs \
    runs/unet_scratch-f1.0 runs/unet_imagenet-f1.0 runs/unet_sentinel2-f1.0 runs/convnet-f1.0

In [ ]:
# @title Save the tables in Drive
!rsync -a outputs/ {DRIVE}/outputs/

In [ ]:
# @title Write pending files to Drive and release the runtime
release_runtime()

## Where the files are

| Path | Contents | Kept after the session |
|---|---|---|
| `MyDrive/garimpo/data.tar.zst` | The data, packed in step 2 | Yes |
| `MyDrive/garimpo/runs/` | Every run: checkpoints, tables, stage statuses | Yes |
| `MyDrive/garimpo/logs/` | The log of every pilot and full-run job | Yes |
| `MyDrive/garimpo/outputs/` | The merged tables of step 5 | Yes |
| `/content/garimpo-uncertainty/` | The code | No |
| `/content/garimpo-data/` | The unpacked data | No |